In [ ]:
# !uv add arviz
# !uv add pymc
# !uv add netCDF4

### if there are problems with the pymce Model widget use
#!uv remove  ipywidgets jupyterlab_widgets -y
#!uv add ipywidgets>=8.0.0 jupyterlab_widgets

In [1]:
import sys
sys.path.insert(1, '/workspaces/aml-working-paper/aml_working_paper/data')

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

import pymc as pm
import arviz as az
from preprocess import preprocess_data
from bayes_hmm import build_hmm_model
from labelswitch import relabel_posterior_states
from routing import *

In [3]:
df_aml = pd.read_csv("/workspaces/aml-working-paper/aml_working_paper/data/synthetic_df_aml.csv")

In [4]:
train_df, val_df, test_df, train_sequences, val_sequences, test_sequences, scaler = preprocess_data(df_aml)

In [5]:
model = build_hmm_model(train_sequences)

In [6]:
### The model configuration should take less than a minute to run in a 8GB CPU and 16RAM computer
### If it takes more, please restart the notebook. You should also be able to see the Pymc widget of the model running
### If not, please go to the top of this notebook and run the respective commands
with model:
    trace_test = pm.sample(
        draws=5,
        tune=5,
        chains=4,
        cores=4,
        target_accept=0.9,
        random_seed=42,
    )

Only 5 samples per chain. Reliable r-hat and ESS diagnostics require longer chains for accurate estimate.
Initializing NUTS using jitter+adapt_diag...
Multiprocess sampling (4 chains in 4 jobs)
NUTS: [pi, current, mu, sigma]


Output()

Sampling 4 chains for 5 tune and 5 draw iterations (20 + 20 draws total) took 1 seconds.
There were 2 divergences after tuning. Increase `target_accept` or reparameterize.
The number of samples is too small to check convergence reliably.


In [8]:
print(
    az.summary(
        trace_test,
        var_names=["pi", "current", "mu", "sigma"],
        round_to=3,
    )
)

                mean     sd  eti89_lb  eti89_ub  ess_bulk  ess_tail  r_hat  \
pi[0]          0.304  0.198     0.097     0.539    19.266    19.266  3.252   
pi[1]          0.295  0.193     0.102     0.576    19.266    19.266  3.536   
pi[2]          0.402  0.107     0.324     0.579    19.266    19.266  3.369   
current[0, 0]  0.419  0.244     0.149     0.793    19.266    19.266  3.258   
current[0, 1]  0.379  0.201     0.117     0.653    19.266    19.266  4.740   
current[0, 2]  0.202  0.152     0.089     0.487    19.266    19.266  4.957   
current[1, 0]  0.228  0.125     0.114     0.385    19.266    19.266  2.581   
current[1, 1]  0.484  0.182     0.303     0.713    19.266    19.266  3.252   
current[1, 2]  0.288  0.190     0.051     0.582    19.266    19.266  3.632   
current[2, 0]  0.144  0.117     0.064     0.407    19.266    19.266  2.312   
current[2, 1]  0.190  0.220     0.034     0.593    19.266    19.266  4.486   
current[2, 2]  0.666  0.267     0.286     0.879    19.266    19.

In [ ]:
with model:
    trace_test = pm.sample(
        draws=100,
        tune=100,
        chains=4,
        cores=4,
        target_accept=0.9,
        random_seed=42,
    )

In [ ]:
print(
    az.summary(
        trace_test,
        var_names=["pi", "current", "mu", "sigma"],
        round_to=3,
    )
)

In [ ]:
print(
    "\nDivergences:",
    trace_test.sample_stats["diverging"].sum().item()
)

In [ ]:
print(
    "\nBFMI:",
    az.bfmi(trace_test)
)

In [ ]:
az.plot_dist(
    trace_test,
    var_names=["mu"],
)

In [ ]:
aligned_trace, permutations = relabel_posterior_states(
    trace_test
)

In [ ]:
import arviz as az

summary_aligned = az.summary(
    aligned_trace,
    var_names=["pi", "current", "mu", "sigma"],
    round_to=3,
)

print(summary_aligned)

In [ ]:
az.plot_trace(
    aligned_trace,
    var_names=["mu"],
);

In [ ]:
for chain in range(permutations.shape[0]):
    unique, counts = np.unique(
        permutations[chain],
        axis=0,
        return_counts=True,
    )

    print(f"\nChain {chain}")

    for perm, count in zip(unique, counts):
        print(
            perm,
            f"{count}/{permutations.shape[1]}"
        )

In [ ]:
with model:
    trace_test = pm.sample(
        draws=500,
        tune=500,
        chains=4,
        cores=4,
        target_accept=0.9,
        random_seed=42,
    )

In [ ]:
np.save(
    "/workspaces/aml-working-paper/aml_working_paper/results/train_account_ids.npy",
    train_df["account_id"].unique()
)

np.save(
    "/workspaces/aml-working-paper/aml_working_paper/results/val_account_ids.npy",
    val_df["account_id"].unique()
)

np.save(
    "/workspaces/aml-working-paper/aml_working_paper/results/test_account_ids.npy",
    test_df["account_id"].unique()
)

In [ ]:
import joblib

joblib.dump(
    scaler,
    "/workspaces/aml-working-paper/aml_working_paper/results/hmm_scaler.joblib"
)

In [ ]:
aligned_trace, permutations = relabel_posterior_states(trace_test)

In [ ]:
trace_test.to_netcdf("/workspaces/aml-working-paper/aml_working_paper/results/bayesian_hmm_trace_test_original.nc")

In [ ]:
aligned_trace.to_netcdf("/workspaces/aml-working-paper/aml_working_paper/results/bayesian_hmm_aligned_trace.nc")

In [ ]:
import arviz as az

summary_aligned = az.summary(
    aligned_trace,
    var_names=["pi", "current", "mu", "sigma"],
    round_to=3,
)

print(summary_aligned)

In [ ]:
plt.rcParams["figure.figsize"] = (15, 15)
az.plot_trace(aligned_trace, var_names=["mu"])


In [ ]:
for chain in range(permutations.shape[0]):
    unique, counts = np.unique(
        permutations[chain],
        axis=0,
        return_counts=True,
    )

    print(f"\nChain {chain}")

    for perm, count in zip(unique, counts):
        print(
            perm,
            f"{count}/{permutations.shape[1]}"
        )

In [ ]:
print(
    "\nDivergences:",
    trace_test.sample_stats["diverging"].sum().item()
)

In [ ]:
print(
    "\nBFMI:",
    az.bfmi(trace_test)
)

In [ ]:
aligned_trace= az.from_netcdf("/workspaces/aml-working-paper/aml_working_paper/results/bayesian_hmm_aligned_trace.nc")

In [ ]:
pi_draws, A_draws, mu_draws, sigma_draws = (extract_posterior_draws(aligned_trace))

In [ ]:
print(pi_draws.shape, A_draws.shape, mu_draws.shape, sigma_draws.shape)

In [ ]:
sequence = val_sequences[0]

results = filter_posterior_draws(
    sequence,
    pi_draws,
    A_draws,
    mu_draws,
    sigma_draws,
)

In [ ]:
print(results["filtered_probs"].shape)
print(results["uncertainty"].shape)
print(results["surprise"].shape)
print(results["information_gain"].shape)

In [ ]:
print(
    results["filtered_probs"]
    .sum(axis=1)[:10]
)

In [ ]:
print(
    results["uncertainty"].min(),
    results["uncertainty"].max(),
)

In [ ]:
print(
    results["parameter_disagreement"].min()
)

In [ ]:
account_id = val_df["account_id"].unique()[0]

In [ ]:
probs = results["filtered_probs"]

In [ ]:
routing_df = pd.DataFrame({
        "account_id": account_id,
        "time": np.arange(len(probs)),

        "p_state_0": probs[:, 0],
        "p_state_1": probs[:, 1],
        "p_state_2": probs[:, 2],

        "uncertainty": results["uncertainty"],
        "surprise": results["surprise"],
        "information_gain": results["information_gain"],

        "expected_state_entropy":
            results["expected_state_entropy"],

        "parameter_disagreement":
            results["parameter_disagreement"],
    })

In [ ]:
routing_df.head()

In [ ]:
plt.rcParams["figure.figsize"] = (10, 5)
plt.plot(routing_df["surprise"], label="suprise")
plt.plot(routing_df["uncertainty"], label= "uncertainty")
plt.legend()

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=True)

axes[0].plot(routing_df["time"], routing_df["surprise"])
axes[0].set_ylabel("Predictive surprise")

axes[1].plot(routing_df["time"], routing_df["uncertainty"])
axes[1].set_ylabel("Normalized entropy")
axes[1].set_xlabel("Time")

plt.tight_layout()
plt.show()

In [ ]:
account_id = routing_df["account_id"].iloc[0]

truth = (
    val_df[val_df["account_id"] == account_id]
    [["time", "aml_state"]]
    .sort_values("time")
)

comparison = routing_df.merge(
    truth,
    on="time",
    validate="one_to_one",
)

comparison["state_transition"] = (
    comparison["aml_state"]
    .ne(comparison["aml_state"].shift())
)

comparison.loc[comparison.index[0], "state_transition"] = False

In [ ]:
transition_times = comparison.loc[
    comparison["state_transition"],
    "time"
].to_numpy()

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=True)

for ax, metric in zip(axes, ["surprise", "uncertainty"]):
    ax.plot(comparison["time"], comparison[metric])

    for t in transition_times:
        ax.axvline(t, color="red", alpha=0.25, linestyle="--")

    ax.set_ylabel(metric)

axes[-1].set_xlabel("Time")
plt.tight_layout()
plt.show()

In [ ]:
validation_routing_df = build_routing_dataframe(
    val_df,
    val_sequences,
    pi_draws,
    A_draws,
    mu_draws,
    sigma_draws,
)

print(validation_routing_df.shape)
print(validation_routing_df.head())

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(9, 6))

plt.scatter(
    validation_routing_df["uncertainty"],
    validation_routing_df["surprise"],
    alpha=0.3,
    s=15,
)

plt.xlabel("Normalized uncertainty")
plt.ylabel("Predictive surprise")
plt.title("Validation accounts: uncertainty vs surprise")

plt.show()

In [ ]:
transition_times

In [ ]:
from scipy.stats import spearmanr, pearsonr

u = validation_routing_df["uncertainty"]
s = validation_routing_df["surprise"]

print("Pearson:", pearsonr(u, s))
print("Spearman:", spearmanr(u, s))

In [ ]:
from routing_investigate import (
    build_candidate_windows,
    select_investigations,
    validate_investigations,
)

candidates = build_candidate_windows(
    validation_routing_df,
    window_size=10,
)

print(candidates.shape)
print(candidates.head())

In [ ]:
strategies = [
    "uncertainty",
    "surprise",
    "combined",
    "random",
]

selected_windows = {}

for strategy in strategies:

    selected = select_investigations(
        candidates,
        budget=10,
        strategy=strategy,
        random_state=42,
    )

    validate_investigations(
        selected,
        candidates,
        budget=10,
    )

    selected_windows[strategy] = selected

    print(f"\nStrategy: {strategy}")
    print(
        selected[
            [
                "account_id",
                "start_time",
                "end_time",
                "routing_score",
            ]
        ].to_string(index=False)
    )

In [ ]:
def selection_overlap(selected_a, selected_b):
    """
    Number of identical selected windows.
    """

    keys_a = set(
        zip(selected_a["account_id"], selected_a["end_time"])
    )

    keys_b = set(
        zip(selected_b["account_id"], selected_b["end_time"])
    )

    return len(keys_a & keys_b)


for i, strategy_a in enumerate(strategies):

    for strategy_b in strategies[i + 1:]:

        overlap = selection_overlap(
            selected_windows[strategy_a],
            selected_windows[strategy_b],
        )

        print(
            f"{strategy_a} vs {strategy_b}: "
            f"{overlap} identical windows"
        )

In [ ]:
print("Validation accounts:", val_df["account_id"].nunique())
print("Routing accounts:", validation_routing_df["account_id"].nunique())
print("Candidate accounts:", candidates["account_id"].nunique())

print("Validation observations:", len(val_df))
print("Candidate windows:", len(candidates))

assert set(candidates["account_id"]) == set(val_df["account_id"])

In [ ]:
def temporal_overlap(selected_a, selected_b):
    """
    Count how many selected windows in A overlap
    at least one selected window in B.
    """

    count = 0

    for _, a in selected_a.iterrows():

        overlaps = selected_b[
            (selected_b["account_id"] == a["account_id"])
            & (selected_b["start_idx"] <= a["end_idx"])
            & (selected_b["end_idx"] >= a["start_idx"])
        ]

        if not overlaps.empty:
            count += 1

    return count

In [ ]:
for i, strategy_a in enumerate(strategies):
    for strategy_b in strategies[i + 1:]:

        overlap = temporal_overlap(
            selected_windows[strategy_a],
            selected_windows[strategy_b],
        )

        print(
            f"{strategy_a} vs {strategy_b}: "
            f"{overlap}/10 windows overlap temporally"
        )